In [1]:
%pip install -U -r requirements.txt -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import boto3
import uuid
import base64
import json
from pathlib import Path
from datetime import date
from boto3.session import Session
from strands import Agent
from strands.models import BedrockModel
from strands.tools import tool

In [3]:
boto_session = Session()
REGION = boto_session.region_name
MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

In [4]:
@tool
def detect_candidate_type(resume_text: str) -> str:
    text = resume_text.lower()

    # Check NURSING first — must be before LOCUMS
    # because "nurse practitioner" contains "nurse"
    is_rn = any(kw in text for kw in [
        "registered nurse", "r.n.", ", rn", "rn,", "rn |",
        "bsn", "msn", "adn", "staff nurse", "rn license"
    ])

    # Only check these if NOT already confirmed RN
    is_physician = not is_rn and any(kw in text for kw in ["m.d.", "physician", "medical doctor"])
    is_pa        = not is_rn and any(kw in text for kw in ["physician assistant", "pa-c", "pa,"])
    is_np        = not is_rn and any(kw in text for kw in ["nurse practitioner", "fnp", "np-c"])
    is_crna      = not is_rn and any(kw in text for kw in ["crna", "nurse anesthetist"])
    is_allied    = not is_rn and any(kw in text for kw in [
        "physical therapist", "surgical tech", "cst", "respiratory therapist",
        "radiologic", "sonographer", "lab tech"
    ])

    if is_physician or is_pa or is_np or is_crna:
        candidate_type = "LOCUMS"
    elif is_rn:
        candidate_type = "NURSING"
    elif is_allied:
        candidate_type = "ALLIED"
    else:
        candidate_type = "UNKNOWN"

    return f"Candidate Type: {candidate_type}"


@tool
def extract_resume_fields(resume_text: str) -> str:
    """
    Extract structured fields from a candidate resume relevant to healthcare job matching.
    Looks for: specialty, recent experience (past 2 years), certifications, education,
    trauma level, teaching hospital, computer charting systems.
    """
    import re
    text = resume_text.lower()
    today = date.today()
    current_year = today.year

    # Recent specialty/role detection (past 2 years)
    recent_years = [str(current_year), str(current_year - 1), str(current_year - 2)]
    recent_context = []
    for line in resume_text.split("\n"):
        if re.search(r'(present|current)', line, re.IGNORECASE):
            recent_context.append(line.strip())
        elif any(yr in line for yr in recent_years):
            recent_context.append(line.strip())

    # Certifications
    cert_keywords = [
        "bls", "acls", "pals", "tncc", "cen", "ccrn", "cnor", "cst", "rrn", "rrt",
        "nrp", "nihss", "stable", "enpc", "atls", "abls", "fbnc", "nicu", "picu",
        "cfrn", "ctrn", "cpen", "board certified", "board eligible", "dea license",
        "state license", "medical license", "rn license", "lpn", "aprn"
    ]
    found_certs = [kw.upper() for kw in cert_keywords if kw in text]

    # Education — specific phrases only to avoid false positives
    edu_keywords = ["associate degree", "bachelor", "bsn", "msn", "adn", "dnp", "phd", "m.d.", "d.o.", "mph"]
    found_edu = [kw.upper() for kw in edu_keywords if kw in text]
    specialty_keywords_resume = [
    "acute care", "acute & critical care", "med surg", "med-surg",
    "medical surgical", "medical-surgical", "icu", "critical care",
    "emergency", "operating room", "labor and delivery"
    ]
    found_specialty = [kw.upper() for kw in specialty_keywords_resume if kw in text]
    # Trauma level
    trauma_level = "Not mentioned"
    if "level i trauma" in text or "level 1 trauma" in text:
        trauma_level = "Level I"
    elif "level ii trauma" in text or "level 2 trauma" in text:
        trauma_level = "Level II"
    elif "level iii trauma" in text or "level 3 trauma" in text:
        trauma_level = "Level III"
    elif "trauma" in text:
        trauma_level = "Trauma mentioned (level unspecified)"

    # Teaching hospital
    teaching = "Not mentioned"
    if any(kw in text for kw in ["teaching hospital", "academic medical", "university hospital", "residency program", "teaching facility"]):
        teaching = "Teaching hospital experience found"

    # Computer charting
    ehr_systems = ["epic", "cerner", "meditech", "allscripts", "eclinicalworks", "athena", "pyxis", "omnicell", "emr", "ehr", "computer charting", "electronic health record", "electronic medical record"]
    found_ehr = [kw.upper() for kw in ehr_systems if kw in text]

    return f"""
=== RESUME FIELD EXTRACTION ===
Recent Experience Context (last 2 years): {recent_context[:5] if recent_context else 'None detected'}
Certifications Found: {', '.join(found_certs) if found_certs else 'None detected'}
Education Keywords: {', '.join(found_edu) if found_edu else 'None detected'}
Trauma Level: {trauma_level}
Teaching Hospital: {teaching}
Computer Charting/EHR Systems: {', '.join(found_ehr) if found_ehr else 'None detected'}
Specialty Detected: {', '.join(found_specialty) if found_specialty else 'None detected'}
"""


@tool
def extract_jd_requirements(jd_text: str) -> str:
    """
    Extract hard and soft requirements from a Job Description.
    Identifies required certifications, licenses, education, specialty, trauma level,
    teaching hospital preference, EHR systems, and candidate type (Nursing/Allied/Locums).
    """
    text = jd_text.lower()

    # Candidate type from JD — NURSING must be checked before LOCUMS
    jd_type = "UNKNOWN"
    if any(kw in text for kw in ["registered nurse", "rn ", "bsn", "msn", "nursing"]):
        jd_type = "NURSING"
    elif any(kw in text for kw in ["physician", "m.d.", "locum", "pa-c", "crna", "nurse practitioner"]):
        jd_type = "LOCUMS"
    elif any(kw in text for kw in ["physical therapist", "occupational", "respiratory", "rad tech", "surgical tech", "lab tech", "sonographer"]):
        jd_type = "ALLIED"

    # Required certs
    import re
    cert_keywords = [
        "bls", "acls", "pals", "tncc", "ccrn", "cnor", "cst", "nrp", "nihss",
        "atls", "enpc", "abls", "board certified", "state license", "rn license", "aprn"
    ]
    # Use word boundary matching to avoid false positives
    # e.g. "dea" matching "ideal", "cen" matching "experienced"
    required_certs = [kw.upper() for kw in cert_keywords if re.search(rf'\b{re.escape(kw)}\b', text)]

    # Education requirements
    edu_keywords = ["associate degree", "bachelor", "bsn", "msn", "adn", "dnp", "phd"]
    required_edu = [kw.upper() for kw in edu_keywords if kw in text]

    if "physician assistant" in text or "pa-c" in text:
        required_edu.append("PA")
    if "doctor of medicine" in text or "m.d." in text:
        required_edu.append("MD")
    if "doctor of osteopathic" in text or "d.o." in text:
        required_edu.append("DO")

    # Trauma level requirement
    trauma_req = "Not specified"
    if "level i trauma" in text or "level 1 trauma" in text:
        trauma_req = "Level I Required"
    elif "level ii trauma" in text or "level 2 trauma" in text:
        trauma_req = "Level II Required"
    elif "level iii trauma" in text or "level 3 trauma" in text:
        trauma_req = "Level III Required"
    elif "trauma" in text:
        trauma_req = "Trauma experience required (level unspecified)"

    # Teaching hospital requirement
    teaching_req = "Not specified"
    if any(kw in text for kw in ["teaching hospital", "academic medical center", "university hospital"]):
        teaching_req = "Teaching hospital experience required"

    # EHR requirements
    ehr_systems = ["epic", "cerner", "meditech", "allscripts", "eclinicalworks", "emr", "ehr", "electronic health record"]
    required_ehr = [kw.upper() for kw in ehr_systems if kw in text]

    # Specialty
    specialty_keywords = [
    "icu", "critical care", "emergency department",
    "operating room", "labor and delivery", "l&d", "oncology",
    "telemetry", "med surg", "med-surg", "medical surg",
    "medical surgical", "medical-surgical",
    "nicu", "picu", "cicu", "cvicu", "stepdown", "step-down",
    "cardiology", "neurology", "orthopedic", "pediatric", "psych",
    "acute care", "acute & critical care", "acute and critical care"
]
    required_specialty = [kw.upper() for kw in specialty_keywords if kw in text]
    # Add this right after the jd_type block
    print("DEBUG JD text sample:", text[:300])
    print("DEBUG jd_type detected:", jd_type)
    print("DEBUG required_certs:", required_certs)

    return f"""
=== JD REQUIREMENTS EXTRACTION ===
Job Type (Nursing/Allied/Locums): {jd_type}
Required Specialty/Unit: {', '.join(set(required_specialty)) if required_specialty else 'Not specified'}
Required Certifications: {', '.join(required_certs) if required_certs else 'None listed'}
Required Education: {', '.join(required_edu) if required_edu else 'None listed'}
Trauma Level Requirement: {trauma_req}
Teaching Hospital Requirement: {teaching_req}
Required EHR/Charting: {', '.join(required_ehr) if required_ehr else 'Not specified'}
"""


@tool
def compute_match_score(
    resume_fields: str,
    jd_requirements: str,
    candidate_type: str
) -> str:
    """
    Compute a detailed match score between resume and JD.
    Scores each dimension: specialty, certifications, education, trauma, teaching hospital, EHR.
    Returns a breakdown and overall match %.
    """
    scores = {}
    notes = []

    # Specialty match — most important
    jd_specialty_line = jd_requirements.split("Required Specialty/Unit:")[-1].split("\n")[0].strip()
    resume_context = resume_fields.lower()
    print("DEBUG jd_specialty_line:", repr(jd_specialty_line))
    print("DEBUG specialty in resume_context:", [spec.strip().lower() for spec in jd_specialty_line.split(", ") if spec.strip().lower() in resume_context])


    if jd_specialty_line == "Not specified":
        scores["Specialty/Primary Role (Recent 2yr)"] = 25
    elif any(spec.strip().lower() in resume_context for spec in jd_specialty_line.lower().split(", ")):
        scores["Specialty/Primary Role (Recent 2yr)"] = 30
    else:
        scores["Specialty/Primary Role (Recent 2yr)"] = 0
        notes.append("Specialty mismatch or not verifiable in recent 2 years")

    # Certifications
    jd_certs_line = jd_requirements.split("Required Certifications:")[-1].split("\n")[0].strip()
    resume_certs_line = resume_fields.split("Certifications Found:")[-1].split("\n")[0].strip()
    if jd_certs_line == "None listed":
        scores["Certifications"] = 20
    else:
        jd_certs = set(c.strip() for c in jd_certs_line.split(","))
        resume_certs = set(c.strip() for c in resume_certs_line.split(","))
        matched = jd_certs & resume_certs
        ratio = len(matched) / max(len(jd_certs), 1)
        scores["Certifications"] = int(20 * ratio)
        if ratio < 1:
            missing = jd_certs - resume_certs
            notes.append(f"Missing certifications: {', '.join(missing)}")

    # Education
    jd_edu_line = jd_requirements.split("Required Education:")[-1].split("\n")[0].strip()
    resume_edu_line = resume_fields.split("Education Keywords:")[-1].split("\n")[0].strip()
    if jd_edu_line == "None listed":
        scores["Education"] = 15
    else:
        jd_edu = set(e.strip() for e in jd_edu_line.split(","))
        resume_edu = set(e.strip() for e in resume_edu_line.split(","))
        if jd_edu & resume_edu:
            scores["Education"] = 15
        else:
            scores["Education"] = 0
            notes.append(f"Education requirement not met: JD needs {jd_edu_line}")

    # Trauma level
    jd_trauma = jd_requirements.split("Trauma Level Requirement:")[-1].split("\n")[0].strip()
    resume_trauma = resume_fields.split("Trauma Level:")[-1].split("\n")[0].strip()
    if jd_trauma == "Not specified":
        scores["Trauma Level"] = 10
    elif jd_trauma.split()[1] in resume_trauma:
        scores["Trauma Level"] = 10
    else:
        scores["Trauma Level"] = 0
        notes.append(f"Trauma level mismatch: JD needs {jd_trauma}, resume shows {resume_trauma}")

    # Teaching hospital
    jd_teach = jd_requirements.split("Teaching Hospital Requirement:")[-1].split("\n")[0].strip()
    resume_teach = resume_fields.split("Teaching Hospital:")[-1].split("\n")[0].strip()
    if jd_teach == "Not specified":
        scores["Teaching Hospital"] = 10
    elif "Teaching hospital experience found" in resume_teach:
        scores["Teaching Hospital"] = 10
    else:
        scores["Teaching Hospital"] = 0
        notes.append("Teaching hospital experience required but not found on resume")

    # EHR/Computer Charting
    jd_ehr_line = jd_requirements.split("Required EHR/Charting:")[-1].split("\n")[0].strip()
    resume_ehr_line = resume_fields.split("Computer Charting/EHR Systems:")[-1].split("\n")[0].strip()
    if jd_ehr_line == "Not specified":
        scores["Computer Charting/EHR"] = 15
    else:
        jd_ehr = set(e.strip() for e in jd_ehr_line.split(","))
        resume_ehr = set(e.strip() for e in resume_ehr_line.split(","))
        if jd_ehr & resume_ehr:
            scores["Computer Charting/EHR"] = 15
        else:
            scores["Computer Charting/EHR"] = 5
            notes.append(f"EHR system mismatch: JD prefers {jd_ehr_line}")

    total = sum(scores.values())
    total = min(total, 95)  # Hard cap — unverifiable claims always exist

    breakdown = "\n".join([f"  {k}: {v}/{[30,20,15,10,10,15][i]}pts" for i, (k, v) in enumerate(scores.items())])

    return f"""
=== MATCH SCORE BREAKDOWN ===
{breakdown}

OVERALL MATCH: {total}/100 ({total}%)

Notes:
{chr(10).join(notes) if notes else 'No major gaps found'}
"""


@tool
def lookup_prior_match_decisions(candidate_type: str, query: str) -> str:
    """
    Search memory for similar past candidate-JD match decisions.
    Args:
        candidate_type: NURSING, ALLIED, or LOCUMS
        query: What to search for (e.g. 'ICU nurse missing CCRN')
    Returns:
        Relevant past decisions from memory
    """
    return f"[Memory lookup — {candidate_type}: '{query}'] No prior decisions yet. Will populate after first runs."

In [5]:
SYSTEM_PROMPT = """You are a senior healthcare staffing specialist reviewing candidate resumes against job descriptions.

Today's date: {today}

Your job is to evaluate match quality across these dimensions:
1. Primary Specialty / Role (focus on past 2 years of experience)
2. Certifications (must match JD-required certs)
3. Education (match degree level if JD specifies)
4. Trauma Level (Level I/II/III — must match if JD requires)
5. Teaching Hospital (flag if JD requires and candidate lacks)
6. Computer Charting / EHR Systems (prefer exact system match)
7. Overall JD Match (hardline basic requirements)

Candidate Type Rules:
- LOCUMS (Physicians, PAs, NPs, CRNAs): Focus on board certification, DEA license, state medical license, specialty board, malpractice history
- NURSING (RNs, LPNs, APRNs): Focus on RN license, BLS/ACLS/specialty certs, unit-specific experience
- ALLIED (PT, OT, RT, CST, Rad Tech, etc.): Focus on profession-specific certifications (CST, RRT, ARRT, etc.), licensure

Scoring (100 points total):
- Specialty/Role (recent 2yr): 30 pts
- Certifications: 20 pts
- Education: 15 pts
- Trauma Level: 10 pts
- Teaching Hospital: 10 pts
- Computer Charting/EHR: 15 pts

After using all tools, provide your analysis.

You MUST end every response with exactly:

MATCH SCORE: XX%
RECOMMENDATION: STRONG MATCH / MODERATE MATCH / WEAK MATCH / NO MATCH
REASON: One clear sentence (max 20 words)
KEY GAPS: List any critical missing requirements

A score of 100% is never valid. Every candidate has at least one gap or 
unverifiable claim. Maximum realistic score is 97%. If your calculation 
reaches 100%, deduct 5 points and document what could not be verified.
""".format(today=date.today().strftime("%B %d, %Y"))

In [6]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

REVIEWER_ID = "staffing-reviewer-001"
memory_manager = MemoryManager(region_name=REGION)

memory = memory_manager.get_or_create_memory(
    name="StaffingMatchMemory_v1",
    strategies=[
        {
            StrategyType.SEMANTIC.value: {
                "name": "PastMatchDecisions",
                "description": "Stores past resume-JD match decisions for Nursing, Allied, and Locums placements",
                "namespaces": ["staffing/decisions/{actorId}/semantic/"],
            }
        },
        {
            StrategyType.USER_PREFERENCE.value: {
                "name": "ReviewerPatterns",
                "description": "Captures reviewer preferences on match thresholds and common rejection patterns",
                "namespaces": ["staffing/reviewer/{actorId}/preferences/"],
            }
        },
    ]
)

memory_id = memory["id"]
print(f"Memory ID: {memory_id}")

# Seed past decisions
memory_client = MemoryClient(region_name=REGION)

past_decisions = [
    ("ICU RN resume, has CCRN, 3 years CVICU, Level II trauma, EPIC charting. JD: CICU RN, CCRN required, Level II, EPIC.", "USER"),
    ("MATCH SCORE: 92%. STRONG MATCH. All certifications met, trauma level matches, EHR matches.", "ASSISTANT"),

    ("Resume: Med-Surg RN, BLS only. JD: ICU RN, CCRN required, ACLS required, 2yr ICU experience.", "USER"),
    ("MATCH SCORE: 25%. NO MATCH. Missing CCRN, ACLS, and ICU experience — critical gaps.", "ASSISTANT"),

    ("Surgical Tech resume, CST certified, 5yr OR experience, Cerner. JD: CST, OR experience, Meditech preferred.", "USER"),
    ("MATCH SCORE: 78%. MODERATE MATCH. CST meets cert req, OR experience strong, EHR system differs.", "ASSISTANT"),

    ("Physician resume: BC Internal Medicine, active DEA, state license. JD: Locums Hospitalist, IM board certified.", "USER"),
    ("MATCH SCORE: 95%. STRONG MATCH. Board certified, fully licensed, specialty aligns.", "ASSISTANT"),

    ("RN resume: Labor & Delivery 4yr, BLS, NRP, Cerner. JD: L&D RN, BLS + NRP required, Epic preferred.", "USER"),
    ("MATCH SCORE: 82%. STRONG MATCH. All required certs present, EHR minor gap only.", "ASSISTANT"),
]

memory_client.create_event(
    memory_id=memory_id,
    actor_id=REVIEWER_ID,
    session_id="seed-session-001",
    messages=past_decisions,
)
print("Seeded past match decisions into memory")

✅ MemoryManager initialized for region: us-west-2
Memory already exists. Using existing memory ID: StaffingMatchMemory_v1-P3DO5iGp6O
🔎 Retrieving memory resource with ID: StaffingMatchMemory_v1-P3DO5iGp6O...
  Found memory: StaffingMatchMemory_v1-P3DO5iGp6O
Existing {'type': 'SEMANTIC', 'name': 'PastMatchDecisions', 'description': 'Stores past resume-JD match decisions for Nursing, Allied, and Locums placements', 'namespaces': ['staffing/decisions/{actorId}/semantic/']}
Requested {'type': 'SEMANTIC', 'name': 'PastMatchDecisions', 'description': 'Stores past resume-JD match decisions for Nursing, Allied, and Locums placements', 'namespaces': ['staffing/decisions/{actorId}/semantic/']}
Existing {'type': 'USER_PREFERENCE', 'name': 'ReviewerPatterns', 'description': 'Captures reviewer preferences on match thresholds and common rejection patterns', 'namespaces': ['staffing/reviewer/{actorId}/preferences/']}
Requested {'type': 'USER_PREFERENCE', 'name': 'ReviewerPatterns', 'description': 'Ca

Memory ID: StaffingMatchMemory_v1-P3DO5iGp6O
Seeded past match decisions into memory


In [7]:
model = BedrockModel(model_id=MODEL_ID, region_name=REGION,temperature=0.0)

memory_config = AgentCoreMemoryConfig(
    memory_id=memory_id,
    session_id=str(uuid.uuid4()),
    actor_id=REVIEWER_ID,
    retrieval_config={
        "staffing/decisions/{actorId}/semantic/": RetrievalConfig(top_k=3, relevance_score=0.25),
        "staffing/reviewer/{actorId}/preferences/": RetrievalConfig(top_k=2, relevance_score=0.2),
    }
)

agent = Agent(
    model=model,
    session_manager=AgentCoreMemorySessionManager(memory_config, REGION),
    tools=[
        detect_candidate_type,
        extract_resume_fields,
        extract_jd_requirements,
        compute_match_score,
        lookup_prior_match_decisions,
    ],
    system_prompt=SYSTEM_PROMPT,
)

print("Staffing Match Agent ready")

Staffing Match Agent ready


In [9]:
import base64
import json
import mimetypes
from pathlib import Path


def extract_text_from_file(file_path: str) -> str:
    """Extract text from PDF (using OCR) or TXT file"""
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    suffix = path.suffix.lower()

    if suffix == '.txt':
        return path.read_text(encoding='utf-8', errors='ignore')
    elif suffix == '.pdf':
        return ocr_pdf(file_path)
    elif suffix in ['.png', '.jpg', '.jpeg', '.gif', '.webp']:
        return ocr_image(file_path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")


def ocr_pdf(file_path: str) -> str:
    """Extract text from PDF using Claude + Vision"""
    path = Path(file_path)
    b64 = base64.b64encode(path.read_bytes()).decode('utf-8')

    content = [
        {
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": b64
            }
        },
        {
            "type": "text",
            "text": "Extract ALL text from this PDF document exactly as it appears. Return only the raw extracted text, nothing else."
        }
    ]

    response = bedrock_runtime.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 4000,
            "messages": [{"role": "user", "content": content}]
        })
    )

    result = json.loads(response['body'].read())
    return result['content'][0]['text']


def ocr_image(file_path: str) -> str:
    """Extract text from image files"""
    path = Path(file_path)
    mime = mimetypes.guess_type(file_path)[0] or 'image/jpeg'
    b64 = base64.b64encode(path.read_bytes()).decode('utf-8')

    content = [
        {
            "type": "image",
            "source": {
                "type": "base64",
                "media_type": mime,
                "data": b64
            }
        },
        {
            "type": "text",
            "text": "Extract ALL text from this image exactly as it appears. Return only the raw text, nothing else."
        }
    ]

    response = bedrock_runtime.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 2000,
            "messages": [{"role": "user", "content": content}]
        })
    )

    result = json.loads(response['body'].read())
    response_text = result['content'][0]['text']

    if "manual review" in response_text.lower() or "tool issues" in response_text.lower():
        print("WARNING: Agent overrode tool output — review tool logs above for failures")

    return response_text


RESUME_PATH = "./resumes/Resume-1.pdf"
JD_PATH = "./jds/JD-3.txt"

print(f"Resume: {RESUME_PATH}")
print(f"Job Description: {JD_PATH}")
print("Extracting text...\n")

try:
    resume_text = extract_text_from_file(RESUME_PATH)
    jd_text = extract_text_from_file(JD_PATH)
    print("Text extracted successfully!\n")
except Exception as e:
    print(f"Error during text extraction: {e}")
    raise


prompt = f"""Please evaluate this candidate against the job description.

=== RESUME TEXT ===
{resume_text}

=== JOB DESCRIPTION TEXT ===
{jd_text}

Use all available tools to:
1. Detect candidate type (Nursing/Allied/Locums)
2. Extract key resume fields
3. Extract JD requirements
4. Compute match score (0-100)
5. Look up any similar past decisions from memory if possible

Then give your final recommendation with clear reasoning."""

response = agent(prompt)

print("\n" + "="*70)
print("FINAL MATCH REPORT")
print("="*70)

if hasattr(response, 'message'):
    content = response.message.get('content', [])
    for block in content:
        if isinstance(block, dict) and block.get('type') == 'text':
            print(block.get('text', ''))
        elif isinstance(block, str):
            print(block)
else:
    print(str(response))

Resume: ./resumes/Resume-1.pdf
Job Description: ./jds/JD-2.txt
Extracting text...

Text extracted successfully!



I'll evaluate Alex Rivera's resume against the Telemetry Unit job description using all available tools.
Tool #6: detect_candidate_type

Tool #7: extract_resume_fields

Tool #8: extract_jd_requirements
DEBUG JD text sample: job title: registered nurse (rn) – telemetry unit
location: chicago, il
department: cardiac telemetry / step-down
employment type: full-time

job summary:
we are seeking a skilled registered nurse with cardiac telemetry experience to join our step-down unit. the ideal candidate will have hands-on ex
DEBUG jd_type detected: NURSING
DEBUG required_certs: ['BLS', 'ACLS', 'TNCC', 'CCRN', 'RN LICENSE']

Tool #9: compute_match_score
DEBUG jd_specialty_line: 'TELEMETRY, STEP-DOWN'
DEBUG specialty in resume_context: []

Tool #10: lookup_prior_match_decisions
## Analysis

Based on my comprehensive evaluation, Alex Rivera presents significant gaps for the Telemet